Nikolaj Skou-Larsen - gkm406

In [38]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from maketables import dtable
from lets_plot import *
LetsPlot.setup_html()
df = pd.read_stata("A1_kommune.dta")

df.columns


Index(['nr', 'kommune', 'taxrev', 'taxrate', 'pop'], dtype='str')

Describtive analysis taxrev total

In [84]:
#Describtive analysis
vars =['taxrev','taxrate','pop']
print(df.groupby('kommune')[vars].mean().round(3))

# Max/min taxrev and taxrate
max_tax_rev = df.loc[df['taxrev'].idxmax()]

min_tax_rev = df.loc[df['taxrev'].idxmin()]

max_tax_rate = df.loc[df['taxrate'].idxmax()]

min_tax_rate = df.loc[df['taxrate'].idxmin()]

selected = pd.DataFrame([
    max_tax_rev,
    min_tax_rev,
    max_tax_rate,
    min_tax_rate
])
selected = selected.rename(columns={
    'taxrev': 'taxrev (mio. DKK)',
    'taxrate': 'taxrate (%)',
    'pop': 'population'
})
selected['kommune'] = [
    'Max tax revenue: ' + str(max_tax_rev['kommune']),
    'Min tax revenue: ' + str(min_tax_rev['kommune']),
    'Max tax rate: ' + str(max_tax_rate['kommune']),
    'Min tax rate: ' + str(min_tax_rate['kommune'])
]
selected['kommune'] = selected['kommune'].str.replace(' Kommune', '', regex=False)



dtable.DTable(
    selected,
    ['taxrev (mio. DKK)', 'taxrate (%)', 'population'],
    counts_row_below=False,
    bycol=['kommune'],
    stats=['mean'],
    caption="Table 1 - Descriptive Statistics"
)

                               taxrev    taxrate       pop
kommune                                                   
Aabenraa Kommune          4547.832031  25.400000   59978.0
Aalborg Kommune          16330.091797  25.400000  197426.0
Aarhus Kommune           26152.337891  24.400000  306650.0
Albertslund Kommune       2964.559082  24.600000   27730.0
Allerød Kommune           1613.119995  25.299999   24089.0
...                               ...        ...       ...
Vejle Kommune             8517.554688  23.400000  106383.0
Vesthimmerlands Kommune   3324.785889  27.200001   38106.0
Viborg Kommune            6772.346191  25.799999   93310.0
Vordingborg Kommune       3733.774902  24.900000   46319.0
Ærø Kommune                566.492981  26.100000    6679.0

[98 rows x 3 columns]


<maketables.mtable.MTable.__repr__.<locals>.DualOutput at 0x24ab01e6270>

Describtive analysis taxrev per person

In [85]:

#Describtive analysis
vars =['taxrev','taxrate','pop']
print(df.groupby('kommune')[vars].mean().round(3))
df['taxrev_per_person'] = (df['taxrev'] * 1_000_000) / df['pop']


# Max/min taxrev and taxrate
max_tax_rev = df.loc[df['taxrev_per_person'].idxmax()]
min_tax_rev = df.loc[df['taxrev_per_person'].idxmin()]
max_tax_rate = df.loc[df['taxrate'].idxmax()]
min_tax_rate = df.loc[df['taxrate'].idxmin()]

# Creats dataframe for only values in Table
selected = pd.DataFrame([
    max_tax_rev,
    min_tax_rev,
    max_tax_rate,
    min_tax_rate
])
# Renames coloumns for readablitty 
selected = selected.rename(columns={
    'taxrev': 'tax revenue per person',
    'taxrate': 'taxrate (%)',
    'pop': 'population'
})
# Adds which category qualifies it for the tabel
selected['kommune'] = [
    'Max revenue: ' + str(max_tax_rev['kommune']),
    'Min revenue: ' + str(min_tax_rev['kommune']),
    'Max rate: ' + str(max_tax_rate['kommune']),
    'Min rate: ' + str(min_tax_rate['kommune'])
]
# Removes Kommune from the table
selected['kommune'] = selected['kommune'].str.replace(' Kommune', '', regex=False)

#Creates the Table
dtable.DTable(
    selected,
    ['tax revenue per person', 'taxrate (%)', 'population'],
    counts_row_below=True,
    bycol=['kommune'],
    stats=['mean'],
    caption="Table 2 - Maximum & minimum values"
)



                               taxrev    taxrate       pop
kommune                                                   
Aabenraa Kommune          4547.832031  25.400000   59978.0
Aalborg Kommune          16330.091797  25.400000  197426.0
Aarhus Kommune           26152.337891  24.400000  306650.0
Albertslund Kommune       2964.559082  24.600000   27730.0
Allerød Kommune           1613.119995  25.299999   24089.0
...                               ...        ...       ...
Vejle Kommune             8517.554688  23.400000  106383.0
Vesthimmerlands Kommune   3324.785889  27.200001   38106.0
Viborg Kommune            6772.346191  25.799999   93310.0
Vordingborg Kommune       3733.774902  24.900000   46319.0
Ærø Kommune                566.492981  26.100000    6679.0

[98 rows x 3 columns]


<maketables.mtable.MTable.__repr__.<locals>.DualOutput at 0x24ab01e6270>

In [33]:
#OLS for log-level

df['log_taxrev'] = np.log(df.taxrev)


df['const'] = 1


result = sm.OLS(df['log_taxrev'], df[['const','taxrate']]).fit()

print(result.summary())  

                            OLS Regression Results                            
Dep. Variable:             log_taxrev   R-squared:                       0.029
Model:                            OLS   Adj. R-squared:                  0.018
Method:                 Least Squares   F-statistic:                     2.818
Date:                Sat, 12 Sep 2026   Prob (F-statistic):             0.0965
Time:                        21:52:31   Log-Likelihood:                -111.12
No. Observations:                  98   AIC:                             226.2
Df Residuals:                      96   BIC:                             231.4
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const         11.6982      2.143      5.459      0.0

Negativ beta1 kan type på at højere skat får folk til at arbejde mindere og dermed bliver skatteindtægterne lavere. Laffer kurven? 